# 1. Data preparation

Part 1 of 5. The series asks whether weak heat pumps (annual SPF H4 below 2.5) can be identified from data that is available without heat metering: design and installation records, whole-system electricity and outdoor temperature.

This notebook:
- reads the summary file and selects 428 air-source heat pumps
- checks the SPF definition
- adds design and installation data
- cleans daily electricity within each home's official 12-month window
- saves two files for notebook 2

Outputs go to `EoH_project/q1_paper` on Google Drive. Runtime is about 2 minutes.

## Step 1: Mount Drive and create the output folder

The project folder `EoH_project` should contain `summary.csv`, `property.csv` and `eoh_cleaned_daily.csv`.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/EoH_project'      # the raw EoH files
OUT_DIR = DATA_DIR + '/q1_paper'                     # everything this paper series makes
FIG_DIR = OUT_DIR + '/figures'
TABLE_DIR = OUT_DIR + '/tables'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)


for name in sorted(os.listdir(DATA_DIR)):
    path = os.path.join(DATA_DIR, name)
    if os.path.isfile(path):
        print(f'{name:40s} {os.path.getsize(path) / 1e6:9.1f} MB')

## Step 2: Libraries and plot style

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10, 'legend.fontsize': 8,
                     'axes.spines.top': False, 'axes.spines.right': False})
BLUE = '#2f6f9f'
ORANGE = '#c96a14'
GREY = '#8a8f98'
GREEN = '#3f8f5a'
MAKER = {'Daikin': 'Manufacturer A', 'Mitsubishi': 'Manufacturer B', 'Vaillant': 'Manufacturer C'}
MAKER_COLOR = {'Daikin': '#2f6f9f', 'Mitsubishi': '#8a5aa8', 'Vaillant': '#5b8c5a'}

print('ready | pandas', pd.__version__)

## Step 3: Summary file

One row per home (742 homes). Key columns:
- `Property_ID`: home identifier
- `HP_Installed`: system type (standard air source, high-temperature air source, ground source or hybrid)
- `Included_SPF_analysis`: whether the home passed quality checks and has a valid annual SPF
- `SPFH4_selected_window`: measured annual SPF, the target

In [ ]:
summary = pd.read_csv(DATA_DIR + '/summary.csv')
print('homes:', summary.shape[0], '| columns:', summary.shape[1])

pd.crosstab(summary['HP_Installed'], summary['Included_SPF_analysis'], margins=True)

## Step 4: Select the study homes

Air-source heat pumps (standard and high-temperature) with a valid SPF. Ground-source and hybrid systems are left out: there are few of them and they behave differently.

In [ ]:
is_air_source = summary['HP_Installed'].isin(['ASHP', 'HT_ASHP'])
has_valid_spf = summary['Included_SPF_analysis'] == True

homes = summary[is_air_source & has_valid_spf].copy().reset_index(drop=True)
print('homes in the study:', len(homes))

pd.crosstab(homes['HP_Brand'], homes['HP_Installed'], margins=True)

## Step 5: Measured SPF

SPF at the H4 boundary is total heat delivered divided by all system electricity (heat pump, electric heaters and circulation pump). A heat pump with SPF below 2.5 is labelled weak.

In [ ]:
homes['SPF'] = homes['SPFH4_selected_window']
homes['weak'] = (homes['SPF'] < 2.5).astype(int)

print('mean SPF  :', round(homes['SPF'].mean(), 2))
print('median SPF:', round(homes['SPF'].median(), 2))
print('weak heat pumps:', homes['weak'].sum(), 'of', len(homes), f"= {100 * homes['weak'].mean():.1f}%")

fig = plt.figure(figsize=(7, 3.5))
plt.hist(homes['SPF'], bins=30, color=BLUE, edgecolor='white')
plt.axvline(2.5, color='red', linestyle='--', linewidth=1, label='Weak threshold (SPF = 2.5)')
plt.xlabel('Measured annual SPF (H4 boundary)')
plt.ylabel('Number of homes')
plt.title(f'Measured efficiency of {len(homes)} air-source heat pumps')
plt.legend()
plt.show()

by_maker = homes.groupby('HP_Brand').agg(homes=('SPF', 'size'), median_SPF=('SPF', 'median'), weak=('weak', 'sum'))
by_maker['weak (%)'] = 100 * by_maker['weak'] / by_maker['homes']
by_maker.round(2)

# ---- save + download this figure (comment out the 2 lines below after the first run) ----
fig.savefig(FIG_DIR + '/q1_01_step05_spf_distribution.png', dpi=300, bbox_inches='tight')
files.download(FIG_DIR + '/q1_01_step05_spf_distribution.png')

## Step 6: Check the SPF definition

SPF is recalculated as (heat pump heat + electric heater electricity) / whole-system electricity. Points should sit on the red line.

In [ ]:
heat = homes['HP_Energy_Output_selected_window']
heaters = homes['IH_Energy_Consumed_selected_window'].fillna(0) + homes['BUH_Energy_Consumed_selected_window'].fillna(0)
electricity = homes['Whole_System_Energy_Consumed_selected_window']

spf_check = (heat + heaters) / electricity
print('largest difference from the official SPF:', round((spf_check - homes['SPF']).abs().max(), 4))

fig = plt.figure(figsize=(4.5, 4.5))
plt.scatter(homes['SPF'], spf_check, s=8, color=BLUE)
plt.plot([1, 5], [1, 5], color='red', linewidth=1)
plt.xlabel('Official SPF')
plt.ylabel('Recalculated SPF')
plt.title('Checking the target definition')
plt.show()

# ---- save + download this figure (comment out the 2 lines below after the first run) ----
fig.savefig(FIG_DIR + '/q1_01_step06_spf_check.png', dpi=300, bbox_inches='tight')
files.download(FIG_DIR + '/q1_01_step06_spf_check.png')

## Step 7: Design and installation data

Only fields known before installation are taken from the property file (floor area, design heat loss and demand, cylinder size, number of adults and a few others).

Measured fields such as heat delivered and flow temperature stay in the table but are never inputs to the main model. Heat delivered is used only to explain results (notebook 4) and flow temperature only for the value-of-information test (notebook 5).

In [ ]:
prop = pd.read_csv(DATA_DIR + '/property.csv', low_memory=False)
print('property file:', prop.shape)

design_cols = ['Property_ID', 'Delivery_Contractor', 'Total_Floor_Area', 'Bedrooms', 'No_Storeys', 'Wall_Type',
               'MCS_SHLoad', 'MCS_Hloss', 'MCS_SHAnnual', 'MCS_DHWAnnual', 'TS_Water_Size', 'HP_Control_Zones', 'No_Adults']
prop = prop[design_cols]
prop = prop[prop['Property_ID'].isin(homes['Property_ID'])].drop_duplicates('Property_ID')
print('rows kept for our homes:', len(prop))

homes = homes.merge(prop, on='Property_ID', how='left')
check_cols = ['MCS_SCOP', 'MCS_Flow_Temp', 'HP_Size_kW'] + design_cols[1:]
(homes[check_cols].notna().mean() * 100).round(1).to_frame('% of homes filled')

## Step 8: Read the daily data

Daily version of the EoH cleansed data, one row per home per day. Only four columns are read:

`Property_ID, Date, Whole_System_Energy_Consumed, External_Air_Temperature`

Heat output, heater meters and water temperatures are left out on purpose, because a smart electricity meter would not see them. Energy values are cumulative meter readings.

In [ ]:
needed_cols = ['Property_ID', 'Date', 'Whole_System_Energy_Consumed', 'External_Air_Temperature']
daily = pd.read_csv(DATA_DIR + '/eoh_cleaned_daily.csv', usecols=needed_cols)
daily = daily.rename(columns={'Whole_System_Energy_Consumed': 'meter_kWh', 'External_Air_Temperature': 'outside_temp'})
print('rows in the file:', f'{len(daily):,}', '| homes:', daily['Property_ID'].nunique())

daily = daily[daily['Property_ID'].isin(homes['Property_ID'])].copy()
daily['Date'] = pd.to_datetime(daily['Date'], dayfirst=True)
daily = daily.sort_values(['Property_ID', 'Date']).reset_index(drop=True)
print('rows for our homes:', f'{len(daily):,}', '| homes found:', daily['Property_ID'].nunique(), 'of', len(homes))
print('first date:', daily['Date'].min().date(), '| last date:', daily['Date'].max().date())

daily[daily['meter_kWh'].notna()].head()

## Step 9: Meter readings to daily electricity

Daily kWh is today's reading minus yesterday's for the same home. Two cleaning rules:
1. A difference is kept only if the previous reading is exactly one day earlier. After a data gap the difference spans several days.
2. Impossible values are removed: negative, or above 150 kWh per day.

In [ ]:
g = daily.groupby('Property_ID')
daily['elec_kWh'] = g['meter_kWh'].diff()
days_since_last = g['Date'].diff().dt.days

daily.loc[days_since_last != 1, 'elec_kWh'] = np.nan                  # rule 1
bad = (daily['elec_kWh'] < 0) | (daily['elec_kWh'] > 150)             # rule 2
print('impossible values removed:', bad.sum())
daily.loc[bad, 'elec_kWh'] = np.nan
daily = daily.drop(columns='meter_kWh')

print('% of days with a valid electricity value:', round(100 * daily['elec_kWh'].notna().mean(), 1))
daily[['elec_kWh', 'outside_temp']].describe().round(2)

## Step 10: Keep each home's official 12-month window

Official SPF is calculated over a fixed 365-day window for each home, so the inputs use the same window. As a check, summed daily electricity is compared with the official annual figure. Homes with missing days fall slightly below the line.

In [ ]:
homes['start'] = pd.to_datetime(homes['Selected_window_start']).dt.tz_localize(None).dt.normalize()
homes['end'] = pd.to_datetime(homes['Selected_window_end']).dt.tz_localize(None).dt.normalize()

daily = daily.merge(homes[['Property_ID', 'start', 'end']], on='Property_ID', how='left')
in_window = (daily['Date'] >= daily['start']) & (daily['Date'] < daily['end'])
daily = daily[in_window].drop(columns=['start', 'end']).reset_index(drop=True)
print('days inside the 12-month windows:', f'{len(daily):,}')

totals = daily.groupby('Property_ID')['elec_kWh'].sum().rename('elec_daily_kWh')
homes = homes.merge(totals, left_on='Property_ID', right_index=True, how='left')
official = homes['Whole_System_Energy_Consumed_selected_window']
ratio = homes['elec_daily_kWh'] / official
print('our total / official total: median', round(ratio.median(), 3), '| lowest', round(ratio.min(), 3),
      '| homes within 2%:', (abs(ratio - 1) < 0.02).sum(), 'of', len(homes))

fig = plt.figure(figsize=(4.5, 4.5))
plt.scatter(official, homes['elec_daily_kWh'], s=8, color=BLUE)
plt.plot([0, official.max()], [0, official.max()], color='red', linewidth=1)
plt.xlabel('Official annual electricity (kWh)')
plt.ylabel('Sum of daily values (kWh)')
plt.title('Checking the daily conversion')
plt.show()

# ---- save + download this figure (comment out the 2 lines below after the first run) ----
fig.savefig(FIG_DIR + '/q1_01_step10_daily_check.png', dpi=300, bbox_inches='tight')
files.download(FIG_DIR + '/q1_01_step10_daily_check.png')

## Step 11: Data coverage

Share of the 365 days with valid electricity and temperature for each home. No homes are removed here; coverage is kept for a robustness check in notebook 4.

In [ ]:
good_day = daily['elec_kWh'].notna() & daily['outside_temp'].notna()
valid_days = daily[good_day].groupby('Property_ID').size().rename('valid_days')
homes = homes.merge(valid_days, left_on='Property_ID', right_index=True, how='left')
homes['valid_days'] = homes['valid_days'].fillna(0)
homes['coverage_%'] = 100 * homes['valid_days'] / (homes['end'] - homes['start']).dt.days

print('homes below 90% coverage:', (homes['coverage_%'] < 90).sum())
print('homes below 80% coverage:', (homes['coverage_%'] < 80).sum())
homes['coverage_%'].describe().round(1).to_frame('coverage (%)')

## Step 12: One year in one home

What a daily smart meter sees: electricity per day (top) and mean outdoor temperature (bottom). The example is the home with near-median SPF among homes with full data. The winter rise in electricity is the basis of the energy signature in notebook 2.

In [ ]:
with_data = homes[homes['coverage_%'] >= 90]
example_home = with_data.loc[(with_data['SPF'] - with_data['SPF'].median()).abs().idxmin(), 'Property_ID']
one = daily[daily['Property_ID'] == example_home]
example_spf = homes.loc[homes['Property_ID'] == example_home, 'SPF'].iloc[0]

fig = plt.figure(figsize=(10, 4.5))
plt.subplot(2, 1, 1)
plt.plot(one['Date'], one['elec_kWh'], color=BLUE, linewidth=1)
plt.ylabel('Electricity (kWh/day)')
plt.title(f'One year in home {example_home} (measured SPF {example_spf:.2f})')
plt.subplot(2, 1, 2)
plt.plot(one['Date'], one['outside_temp'], color=ORANGE, linewidth=1)
plt.ylabel('Outdoor temp (C)')
plt.tight_layout()
plt.show()

# ---- save + download this figure (comment out the 2 lines below after the first run) ----
fig.savefig(FIG_DIR + '/q1_01_step12_one_year.png', dpi=300, bbox_inches='tight')
files.download(FIG_DIR + '/q1_01_step12_one_year.png')

## Step 13: Save for notebook 2

- `q1_homes.csv`: one row per home with the target, design data and coverage
- `q1_daily.csv`: clean daily electricity and outdoor temperature within each home's window

In [ ]:
homes = homes.drop(columns=['start', 'end'])
homes.to_csv(OUT_DIR + '/q1_homes.csv', index=False)
daily.to_csv(OUT_DIR + '/q1_daily.csv', index=False)
print('saved q1_homes.csv:', homes.shape)
print('saved q1_daily.csv:', daily.shape)